# Figure 9: Regional effect of each drag component on surge

Change in annual-maximum surge between consecutive experiments against the change in air density, wind speed and Charnock parameter, for four regions (NEU, RFE, NEN, EAS).

From Özkan et al. (2025), *Sensitivity of global storm surge modelling to sea surface drag*, Ocean Dynamics, https://doi.org/10.1007/s10236-025-01713-3

The processed input files are not included in this repository; this notebook shows how the figure was made. Experiment names: in the code, `cc` = FIX, `cv` = NST, `ntr`/`uv10n` = CAD and `rho`/`uv10rho` = IFS (see the main README).

In [ ]:
# Folder with the processed input files (not included in this repository)
DATA_DIR = "path/to/paper1_data/"

In [ ]:
import os
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import proplot as pplt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import regionmask
import string
from scipy.stats import gaussian_kde
from numpy.polynomial.polynomial import Polynomial
from scipy.stats import linregress
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [ ]:
data_dir =  DATA_DIR + "data/4-Processed_for_Visualization/2_global_statistics/Aligned_files/"

dataset_paths = {
    "gtsm_cc": data_dir + "1-CC_Annualmax.nc",
    "gtsm_cv": data_dir + "2-CV_Aligned_to_annualmax_cc.nc",
    "gtsm_ntr": data_dir + "3-NTR_Aligned_to_annualmax_cc.nc",
    "gtsm_rho": data_dir + "4-RHO_Aligned_to_annualmax_cc.nc",
    "era5_c": data_dir + "ERA5_C_Aligned_to_annualmax_cc.nc",
    "era5_wsdiff": data_dir + "ERA5_WSDIFF_Aligned_to_annualmax_cc.nc",
    "era5_rho": data_dir + "ERA5_RHO_Aligned_to_annualmax_cc.nc"
}

In [ ]:
# Initialize dictionaries to store datasets
model_data = {}
era5_data = {}

# Function to adjust dataset
def adjust_model_data(dataset):
    # Rename the "year" dimension to "time" and convert to datetime
    dataset = dataset.rename({"date": "time"})
    dataset = dataset.assign_coords(time=pd.to_datetime(dataset.time.values, format='%Y'))

    # Rename latitude and longitude, if present
    if 'longitude' in dataset.coords:
        dataset = dataset.rename({'longitude': 'lon'})
    if 'latitude' in dataset.coords:
        dataset = dataset.rename({'latitude': 'lat'})

    # Assign necessary coordinates
    coords_to_assign = {name: dataset[name] for name in ['lon', 'lat', 'time', 'station_name'] if name in dataset}
    dataset = dataset.assign_coords(**coords_to_assign)

    # Sort by "station_name" if available
    if "station_name" in dataset:
        dataset = dataset.sortby("station_name")

    return dataset

# Load and adjust each dataset
for name, path in dataset_paths.items():
    dataset = xr.open_dataset(path)
    adjusted_dataset = adjust_model_data(dataset)
    
    # Store in appropriate dictionary based on dataset type
    if name.startswith("era5"):
        era5_data[name] = adjusted_dataset
    else:
        model_data[name] = adjusted_dataset

In [ ]:
# Function to apply regional adjustments
def apply_regional_adjustments(annual_max_datasets):
    regional_annual_max_datasets = {}
    for sim_key in annual_max_datasets.keys():
        mask_3D_annual = regionmask.defined_regions.ar6.land.mask_3D(annual_max_datasets[sim_key], drop=False)
        regional_annual_max_datasets[sim_key] = mask_3D_annual
    return regional_annual_max_datasets

# Apply regional adjustments to both model and ERA5 datasets
regional_model_data = apply_regional_adjustments(model_data)
regional_era5_data = apply_regional_adjustments(era5_data)

# Define AR6 region mapping
ar6_land = regionmask.defined_regions.ar6.land
ar6_land_mapping = dict(zip(ar6_land.abbrevs, ar6_land.names))

# Function to process data for a specific region for model
def process_dataset(regional_dataset, original_datasets, model_key, region_abbrev):
    selected_region = regional_dataset[model_key].isel(region=(regional_dataset[model_key].abbrevs == region_abbrev))
    dmax_sim = original_datasets[model_key].where(selected_region)
    df_sim = dmax_sim.to_dataframe().reset_index()
    
    # Select 'time' and all columns that start with 'max_'
    max_columns = [col for col in df_sim.columns if col.startswith('max_')]
    df_sim = df_sim[['time'] + max_columns]
    
    return df_sim

In [ ]:
# Get AR6 land regions and map abbreviations to names
ar6_land = regionmask.defined_regions.ar6.land
ar6_land_mapping = dict(zip(ar6_land.abbrevs, ar6_land.names))
region_mapping = ar6_land_mapping

In [ ]:
# Define regions and corresponding diverging colormaps
regions = ["NEU", "RFE", "NEN", "EAS"]

"""
region_colormaps = {
    "NEU": "div",  # Diverging colormap for NEU
    "RFE": "Curl",      # Diverging colormap for RFE
    "NEN": "Delta",       # Diverging colormap for NEN
    "EAS": "PiYg"       # Diverging colormap for EAS
}
"""

region_colormaps = {
    "NEU": "jet",  # Diverging colormap for NEU
    "RFE": "jet",      # Diverging colormap for RFE
    "NEN": "jet",       # Diverging colormap for NEN
    "EAS": "jet"       # Diverging colormap for EAS
}

# Initialize plotting
fig, axes = pplt.subplots(nrows=4, ncols=3, sharey=True, sharex=True, figwidth=10, spanx=True, space=2.00)
fig.format(
    ylabel='Surge height difference (m)', 
    xlabel='', ylabelsize=12, xlabelsize=12, grid=True,
    abcloc='ul', abc='a', abcbbox=True, abcsize=12,
    leftlabels=[f"{region_abbrev}—{region_mapping[region_abbrev]}" for region_abbrev in regions],
                leftlabelweight='bold', leftlabelsize=12,
    toplabels=['$GTSM_{CAD}$—$GTSM_{IFS}$', 
               '$GTSM_{NST}$—$GTSM_{CAD}$', 
               '$GTSM_{FIX}$—$GTSM_{NST}$'],
    toplabelweight='bold', toplabelsize=12,
    bottomlabels=['Air density difference (1.20—$ρ_{a}$) (kg/m³)',
                  'Wind speed difference ($ΔU_{10}$) (m/s)', 
                  'Charnock difference (0.041—$α_{Chvar}$)'],
    bottomlabelweight='normal', bottomlabelsize=10,
    xtickminor=False
)

# Loop over the regions to process and plot data
for i, region_abbrev in enumerate(regions):
    # Assume `region_mapping` dictionary is available for full region names
    region_full_name = region_mapping[region_abbrev]
    
    df_cc = process_dataset(regional_model_data, model_data, "gtsm_cc", region_abbrev)
    df_cv = process_dataset(regional_model_data, model_data, "gtsm_cv", region_abbrev)
    df_ntr = process_dataset(regional_model_data, model_data, "gtsm_ntr", region_abbrev)
    df_rho = process_dataset(regional_model_data, model_data, "gtsm_rho", region_abbrev)
    df_c = process_dataset(regional_era5_data, era5_data, "era5_c", region_abbrev)
    df_wsdiff = process_dataset(regional_era5_data, era5_data, "era5_wsdiff", region_abbrev)
    df_rho_era5 = process_dataset(regional_era5_data, era5_data, "era5_rho", region_abbrev)
    
    #Calculate surge effects
    surge_density_effect = df_ntr['max_surge'] - df_rho['max_surge']
    surge_wind_effect = df_cv['max_surge'] - df_ntr['max_surge']
    surge_charnock_effect = df_cc['max_surge'] - df_cv['max_surge']

    charnock_change = df_c['max_c'] - 0.041
    charnock_change = charnock_change * -1
    wind_diff_change = df_wsdiff['max_wsdiff']
    air_density_change = df_rho_era5['max_rho'] - 1.2
    air_density_change = air_density_change * -1

    # Prepare data frames for each parameter
    df_charnock = pd.DataFrame({'charnock_change': charnock_change.values.flatten(), 'surge_effect': surge_charnock_effect.values.flatten()}).dropna()
    df_wind = pd.DataFrame({'wind_speed_diff': wind_diff_change.values.flatten(), 'surge_effect': surge_wind_effect.values.flatten()}).dropna()
    df_density = pd.DataFrame({'air_density_change': air_density_change.values.flatten(), 'surge_effect': surge_density_effect.values.flatten()}).dropna()

    df_max_surge_cc = pd.DataFrame({'max_surge': df_cc['max_surge'].values.flatten()}).dropna()
    df_max_surge_cv = pd.DataFrame({'max_surge': df_cv['max_surge'].values.flatten()}).dropna()
    df_max_surge_ntr = pd.DataFrame({'max_surge': df_ntr['max_surge'].values.flatten()}).dropna()
    df_max_surge_rho = pd.DataFrame({'max_surge': df_rho['max_surge'].values.flatten()}).dropna()
    
    # Get the diverging colormap for the current region
    cmap = region_colormaps[region_abbrev]
    
    # Plot for air density change
    sc1 = axes[i, 0].scatter(
        df_density['air_density_change'],
        df_density['surge_effect'],
        c=df_max_surge_ntr["max_surge"],
        cmap=cmap, alpha=0.8
    )
    
    # Plot for wind speed difference
    sc2 = axes[i, 1].scatter(
        df_wind['wind_speed_diff'],
        df_wind['surge_effect'],
        c=df_max_surge_cv["max_surge"],
        cmap=cmap, alpha=0.8
    )
    
    # Plot for charnock change
    sc3 = axes[i, 2].scatter(
        df_charnock['charnock_change'],
        df_charnock['surge_effect'],
        c=df_max_surge_cc["max_surge"],
        cmap=cmap, alpha=0.8
    )

    # Add a colorbar for each row, aligned with the last column
    fig.colorbar(sc3, ax=axes[i, -1], label=f'Surge height (m)', loc='r', 
                 shrink=0.9, pad=0.03, labelsize=12, ticklabelsize=10)
    
ch_limits = (-0.08, 0.04)
ws_limits = (-2, 1.55)
ad_limits = (-0.4, 0.2)

sh0_limits = (-0.65, 0.22)
sh1_limits = (-0.4, 0.62)
sh2_limits = (-0.6, 0.8)
sh3_limits = (-0.3, 0.32)

axes[0, 0].format(xlim=ad_limits, ylim=sh0_limits)
axes[0, 1].format(xlim=ws_limits, ylim=sh0_limits)
axes[0, 2].format(xlim=ch_limits, ylim=sh0_limits)
axes[1, 0].format(xlim=ad_limits, ylim=sh1_limits)
axes[1, 1].format(xlim=ws_limits, ylim=sh1_limits)
axes[1, 2].format(xlim=ch_limits, ylim=sh1_limits)
axes[2, 0].format(xlim=ad_limits, ylim=sh2_limits)
axes[2, 1].format(xlim=ws_limits, ylim=sh2_limits)
axes[2, 2].format(xlim=ch_limits, ylim=sh2_limits)
axes[3, 0].format(xlim=ad_limits, ylim=sh3_limits)
axes[3, 1].format(xlim=ws_limits, ylim=sh3_limits)
axes[3, 2].format(xlim=ch_limits, ylim=sh3_limits)

# Save the plot
# Save the plot as PDF and PNG, cropping the bounding box
# Define the target directory
save_dir = 'final_plots'
os.makedirs(save_dir, exist_ok=True)  # Create the directory if it doesn't exist

# Save the plot as PDF and PNG in the specified directory
fig.savefig(os.path.join(save_dir, f'regional_surge_effect_3.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(save_dir, f'regional_surge_effect_3.png'), dpi=600, bbox_inches='tight')

# Adjust the global formatting
fig.show()